# Experiment 3 (prototype). Theory of the optimal sieving bound B*

**Question.** Can B* be predicted from a few platform constants, without a sweep over B?

**Marginal argument.** Add the prime p ≈ B to the sieve. The fraction of candidates that reach this prime is
approximately W(B) = ∏_{q≤B}(1 − 1/q).
- *Additional cost:* one single-precision reduction, c_mod, for trial division; for the segmented prefilter,
  the last segment product grows by log₂p bits, i.e. approximately κ·log₂p (κ is the slope of the GCD cost per bit).
- *Saving:* a fraction 1/p of the candidates no longer reaches Miller–Rabin, i.e. c_MR / p.

At the optimum both are equal and W(B) cancels:

    trial division:        B*_TD  ≈ c_MR / c_mod
    segmented prefilter:   B*_SEG · log₂ B*_SEG ≈ c_MR / κ

Because κ·log₂p ≪ c_mod, B*_SEG ≫ B*_TD: segmentation moves the cost-optimal bound upwards. Since c_MR grows
much faster with the bit length than κ, B* also grows strongly with the bit length.

**Experiment.** For each bit length:
1. measure c_MR, c_mod and the GCD cost c(x) as a function of the product length x;
2. locate the measured B* on a fine logarithmic grid;
3. compare it with the exact cost model (eq. (cseg)/(total) of the paper) and with the closed form above.

The results are written to `results_03/` and `results_03.zip`.

In [ ]:
MODE = "quick"          # "smoke" | "quick" | "full"
SEED = 20261101
P = {
    "smoke": dict(BITS=[512, 1024], N=300, REPS=3, GRID_DEC=(2.5, 5.0, 0.5), MRK=10),
    "quick": dict(BITS=[512, 1024, 2048, 4096], N=1500, REPS=5, GRID_DEC=(2.5, 6.5, 0.25), MRK=20),
    "full":  dict(BITS=[512, 768, 1024, 1536, 2048, 3072, 4096], N=3000, REPS=9, GRID_DEC=(2.5, 7.0, 0.125), MRK=40),
}[MODE]

In [ ]:
import math, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import dtlib as d

R = d.Report("03", MODE, SEED)
R.log(f"Arithmetic backend: {d.BACKEND}")
plt.rcParams.update({"font.family": "DejaVu Serif", "axes.grid": True, "grid.alpha": 0.3})
GRID = [int(round(10 ** e)) for e in np.arange(*P["GRID_DEC"])]
R.log(f"Grid of B: {GRID}")

## 3.1. Platform constants: c_MR, c_mod, GCD cost c(x) and its slope κ

In [ ]:
const_rows, gcd_rows = [], []
for bits in P["BITS"]:
    rnd = random.Random(SEED + bits)
    C = [d.mpz(d.random_odd(bits, rnd)) for _ in range(min(P["N"], 1000))]
    c_mr = d.mr_round_cost(bits, k=P["MRK"])
    c_mod, _ = d.time_per_item(lambda n: n % 997 == 0, C, reps=P["REPS"])
    big = d.odd_primes_upto(2_000_000)
    meas = []
    for k in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]:
        M = d.prod(big[:k])
        t, _ = d.time_per_item(lambda n: d.gcd(n, M), C[:300], reps=P["REPS"])
        meas.append((int(M).bit_length(), t)); gcd_rows.append({"bits": bits, "product_bits": int(M).bit_length(), "gcd_ns": t})
    xb = np.array([m[0] for m in meas], float); yc = np.array([m[1] for m in meas])
    sel = xb >= 2000
    kappa, alpha = np.polyfit(xb[sel], yc[sel], 1)
    const_rows.append({"bits": bits, "c_MR_us": c_mr / 1e3, "c_mod_ns": c_mod, "kappa_ns_per_bit": kappa, "alpha_ns": alpha})
    R.log(f"{bits} bits: c_MR={c_mr / 1e3:.1f} us, c_mod={c_mod:.0f} ns, kappa={kappa:.3f} ns/bit")
df_c = pd.DataFrame(const_rows); df_g = pd.DataFrame(gcd_rows)
R.table(df_c, "T1_constants", "Platform constants")
R.table(df_g, "T2_gcd_cost", "GCD cost as a function of the candidate length and the product length")
GC = {b: (df_g[df_g.bits == b]["product_bits"].values, df_g[df_g.bits == b]["gcd_ns"].values) for b in P["BITS"]}
CONST = df_c.set_index("bits")

def gcd_cost(bits, mb):
    xb, yc = GC[bits]
    if mb <= xb[-1]:
        return float(np.interp(mb, xb, yc))
    c = CONST.loc[bits]
    return float(c["alpha_ns"] + c["kappa_ns_per_bit"] * mb)

## 3.2. Measured B*, B* of the exact cost model, and B* of the closed form

In [ ]:
sweep_rows, res_rows = [], []
for bits in P["BITS"]:
    rnd = random.Random(SEED + 3 * bits)
    C = [d.mpz(d.random_odd(bits, rnd)) for _ in range(P["N"])]
    c = CONST.loc[bits]; c_mr = c["c_MR_us"] * 1e3
    for B in GRID:
        pf = d.SegPrefilter(B)
        t, _ = d.time_per_item(pf.passes, C, reps=max(3, P["REPS"] // 2))
        pi = d.survival(pf.P)
        # exact cost model of the segmented prefilter
        reach, model = 1.0, 0.0
        for blk, mb in zip(pf.blocks, pf.bits):
            model += reach * gcd_cost(bits, mb); reach *= d.survival(blk)
        row = {"bits": bits, "B": B, "segments": len(pf.mods), "W_B": pi, "prefilter_us": t / 1e3,
               "model_prefilter_us": model / 1e3, "expected_cost_us": (t + pi * c_mr) / 1e3,
               "model_expected_cost_us": (model + pi * c_mr) / 1e3}
        if B <= 100_000:
            td = d.TrialDiv(B)
            ttd, _ = d.time_per_item(td.passes, C[:min(len(C), 800)], reps=3)
            row["trial_division_us"] = ttd / 1e3; row["trial_division_expected_cost_us"] = (ttd + pi * c_mr) / 1e3
        sweep_rows.append(row)
    R.log(f"{bits} bits: sweep over B finished")
df_sw = pd.DataFrame(sweep_rows)
R.table(df_sw, "T3_sweep", "Fine sweep over B: measured and modelled costs")

def parabolic_min(B, y):
    """Grid minimum refined by a parabola in log10 B."""
    i = int(np.argmin(y))
    if 0 < i < len(y) - 1:
        x = np.log10(B[i - 1:i + 2]); yy = y[i - 1:i + 2]
        a, b_, _ = np.polyfit(x, yy, 2)
        if a > 0:
            return float(10 ** (-b_ / (2 * a)))
    return float(B[i])

def solve_closed(rhs):          # B log2 B = rhs
    B = max(rhs / 20, 10.0)
    for _ in range(60):
        B = rhs / math.log2(B)
    return B

for bits in P["BITS"]:
    x = df_sw[df_sw.bits == bits].sort_values("B")
    c = CONST.loc[bits]; c_mr = c["c_MR_us"] * 1e3
    B_emp = parabolic_min(x.B.values, x["expected_cost_us"].values)
    B_mod = parabolic_min(x.B.values, x["model_expected_cost_us"].values)
    B_cf = solve_closed(c_mr / c["kappa_ns_per_bit"])
    xt = x.dropna(subset=["trial_division_expected_cost_us"])
    B_td_emp = parabolic_min(xt.B.values, xt["trial_division_expected_cost_us"].values)
    B_td_cf = c_mr / c["c_mod_ns"]
    edge = "yes" if x["expected_cost_us"].values.argmin() == len(x) - 1 else "no"
    res_rows.append({"bits": bits, "Bstar_measured": B_emp, "Bstar_model": B_mod, "Bstar_closed_form": B_cf,
                     "closed_over_measured": B_cf / B_emp, "Bstar_TD_measured": B_td_emp, "Bstar_TD_formula": B_td_cf,
                     "Bstar_over_Bstar_TD": B_emp / B_td_emp, "minimum_at_grid_edge": edge})
df_res = pd.DataFrame(res_rows)
R.table(df_res, "T4_Bstar_compare", "B*: measurement, exact model, and closed forms (B log2 B = c_MR/kappa; B_TD = c_MR/c_mod)")
R.summary["Bstar"] = df_res.to_dict("records")

# scaling law: log B* ~ k log b
if len(df_res) >= 3:
    k_emp = np.polyfit(np.log(df_res.bits), np.log(df_res["Bstar_measured"]), 1)[0]
    k_mr = np.polyfit(np.log(df_c.bits), np.log(df_c["c_MR_us"]), 1)[0]
    k_kappa = np.polyfit(np.log(df_c.bits), np.log(df_c["kappa_ns_per_bit"].clip(lower=1e-6)), 1)[0]
    R.summary["scaling"] = {"k of B* (measured)": k_emp, "k of c_MR": k_mr, "k of kappa": k_kappa,
                            "predicted k = k_MR - k_kappa (without log correction)": k_mr - k_kappa}
    R.table(pd.DataFrame([R.summary["scaling"]]), "T5_scaling", "Scaling laws (power-law fits)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for bits in P["BITS"]:
    x = df_sw[df_sw.bits == bits].sort_values("B")
    rel = x["expected_cost_us"] / x["expected_cost_us"].min()
    l, = ax[0].semilogx(x.B, rel, "o-", ms=3, label=f"{bits} bits")
    ax[0].semilogx(x.B, x["model_expected_cost_us"] / x["expected_cost_us"].min(), ":", color=l.get_color())
ax[0].set_ylim(0.95, 1.6); ax[0].set_xlabel("B"); ax[0].set_ylabel("Expected cost / minimum")
ax[0].set_title("Measurement (solid) and model (dotted)"); ax[0].legend(fontsize=8)
ax[1].loglog(df_res.bits, df_res["Bstar_measured"], "o-", label="B* measured")
ax[1].loglog(df_res.bits, df_res["Bstar_model"], "s--", label="B* model")
ax[1].loglog(df_res.bits, df_res["Bstar_closed_form"], "^:", label="B log2 B = c_MR/kappa")
ax[1].loglog(df_res.bits, df_res["Bstar_TD_measured"], "x-", label="B*_TD measured")
ax[1].loglog(df_res.bits, df_res["Bstar_TD_formula"], "+:", label="B_TD = c_MR/c_mod")
ax[1].set_xlabel("Candidate bit length"); ax[1].set_ylabel("Best B"); ax[1].legend(fontsize=8)
R.figure(fig, "F1_Bstar", "Optimal sieving bound: measurement, model, and closed forms")

In [ ]:
R.finish()